# 01 — Data Cleaning

Documents and walks through the exact cleaning pipeline implemented in [`dataset.py`](../dataset.py), with before/after
statistics so the cleaning decisions are auditable rather than opaque.

Two raw sources are combined:

| Source | File | Records (raw) | Population |
|---|---|---:|---|
| Russia (Kaggle "Cardiovascular Disease") | `cardio_train.csv` | 70,000 | Russian patients |
| China (Shanxi regional) | `shanxi_cardio.csv` | 19,999 | Shanxi province patients |


In [1]:
import pandas as pd
import numpy as np
import os

BASE = os.path.join(os.getcwd(), "..")
pd.set_option("display.max_columns", None)


## 1. Load raw data

In [2]:
russia = pd.read_csv(os.path.join(BASE, "cardio_train.csv"), sep=";")
china  = pd.read_csv(os.path.join(BASE, "shanxi_cardio.csv"))

print("Russia raw:", russia.shape)
print("China raw :", china.shape)
russia.head()


Russia raw: (70000, 13)
China raw : (19999, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [3]:
china.head()


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,1,20540,1,170,72.0,120,80,2,1,0,0,1,0
1,2,22431,1,163,72.0,135,80,1,2,0,0,0,1
2,3,19066,2,183,105.0,180,90,3,1,0,1,0,1
3,4,22601,1,158,126.0,140,90,2,2,0,0,1,1
4,5,19240,2,168,76.0,120,80,1,1,1,0,1,0


## 2. Check for missing values and duplicate ids

Neither raw source has missing values, but both carry an `id` column that is dropped before modeling since it carries
no predictive signal and differs in scheme between the two sources.

In [4]:
print("Russia nulls:\n", russia.isna().sum().sum())
print("China nulls:\n", china.isna().sum().sum())
print("Russia duplicate rows (excl. id):", russia.drop(columns=["id"]).duplicated().sum())
print("China duplicate rows (excl. id):", china.drop(columns=["id"]).duplicated().sum())


Russia nulls:
 0
China nulls:
 0
Russia duplicate rows (excl. id): 24
China duplicate rows (excl. id): 1


In [5]:
russia = russia.drop(columns=["id"], errors="ignore")
china  = china.drop(columns=["id"], errors="ignore")

russia["source"] = "russia"
china["source"]  = "china"


## 3. Feature engineering

- `age` is recorded in **days** in both raw sources — converted to whole years.
- `bmi` is derived from `weight` (kg) and `height` (cm): `weight / (height/100)^2`.


In [6]:
russia["age"] = (russia["age"] / 365).round().astype(int)
china["age"]  = (china["age"]  / 365).round().astype(int)

russia["bmi"] = russia["weight"] / ((russia["height"] / 100) ** 2)
china["bmi"]  = china["weight"]  / ((china["height"]  / 100) ** 2)

cols = ["age","gender","height","weight","bmi","ap_hi","ap_lo",
        "cholesterol","gluc","smoke","alco","active","cardio","source"]
russia = russia[cols]
china  = china[cols]

combined_raw = pd.concat([russia, china], ignore_index=True)
combined_raw.describe().T


,count,mean,std,min,25%,50%,75%,max
age,89999.0,53.344182,6.763965,30.000000,48.000000,54.000000,58.000000,65.000000
gender,89999.0,1.350982,0.477280,1.000000,1.000000,1.000000,2.000000,2.000000
height,89999.0,164.354815,8.221839,55.000000,159.000000,165.000000,170.000000,250.000000
weight,89999.0,74.191736,14.434708,10.000000,65.000000,72.000000,82.000000,200.000000
bmi,89999.0,27.550204,6.047648,3.471784,23.875115,26.365603,30.222222,298.666667
ap_hi,89999.0,128.857665,153.764739,-150.000000,120.000000,120.000000,140.000000,16020.000000
ap_lo,89999.0,96.435572,181.625641,-70.000000,80.000000,80.000000,90.000000,11000.000000
cholesterol,89999.0,1.368737,0.681942,1.000000,1.000000,1.000000,2.000000,3.000000
gluc,89999.0,1.226747,0.572368,1.000000,1.000000,1.000000,1.000000,3.000000
smoke,89999.0,0.088079,0.283411,0.000000,0.000000,0.000000,0.000000,1.000000


## 4. Outlier / implausible-value filtering

Self-reported clinical intake data (particularly blood pressure) contains values that are not physiologically
possible — e.g. `ap_hi` of 16,020 or negative diastolic pressure. Rather than clip these, rows are **dropped** so
the training distribution reflects real, plausible clinical ranges:

| Field | Valid range | Rationale |
|---|---|---|
| `ap_hi` (systolic) | 70–250 mmHg | Below/above is incompatible with life or a data entry error (e.g. missing decimal) |
| `ap_lo` (diastolic) | 40–150 mmHg | Same |
| `ap_hi > ap_lo` | — | Systolic must exceed diastolic by definition |
| `height` | 100–220 cm | Excludes child-height / data-entry errors in an adult dataset |
| `weight` | 30–200 kg | Excludes implausible entries |
| `bmi` | 10–60 | Excludes compounding errors in height/weight |


In [7]:
def clean_cardio(df):
    df = df.copy()
    before = len(df)
    df = df[df["ap_hi"]  >= 70]
    df = df[df["ap_hi"]  <= 250]
    df = df[df["ap_lo"]  >= 40]
    df = df[df["ap_lo"]  <= 150]
    df = df[df["ap_hi"]  >  df["ap_lo"]]
    df = df[df["height"] >= 100]
    df = df[df["height"] <= 220]
    df = df[df["weight"] >= 30]
    df = df[df["weight"] <= 200]
    df = df[(df["bmi"] >= 10) & (df["bmi"] <= 60)]
    after = len(df)
    print(f"  {before:>7,} -> {after:>7,}  ({(before-after)/before:.2%} dropped)")
    return df.reset_index(drop=True)

print("Russia:")
russia_clean = clean_cardio(russia)
print("China:")
china_clean = clean_cardio(china)
print("Combined:")
combined_clean = clean_cardio(combined_raw)


Russia:
   70,000 ->  68,598  (2.00% dropped)
China:
   19,999 ->  19,604  (1.98% dropped)
Combined:
   89,999 ->  88,202  (2.00% dropped)


## 5. Cholesterol / glucose category → clinical value mapping

The source datasets encode cholesterol and glucose as ordinal categories (1/2/3). These are mapped to representative mg/dL midpoints and human-readable labels for reporting and visualization.

In [8]:
CHOLESTEROL_MAP = {1: 180, 2: 220, 3: 270}
GLUCOSE_MAP     = {1:  90, 2: 115, 3: 180}
chol_labels = {1: "Normal (<200)", 2: "Above Normal (200-239)", 3: "Well Above Normal (>=240)"}
gluc_labels = {1: "Normal (<100)", 2: "Above Normal (100-125)", 3: "Well Above Normal (>=126)"}

for df in [russia_clean, china_clean, combined_clean]:
    df["cholesterol_mgdl"]   = df["cholesterol"].map(CHOLESTEROL_MAP)
    df["gluc_mgdl"]          = df["gluc"].map(GLUCOSE_MAP)
    df["cholesterol_label"]  = df["cholesterol"].map(chol_labels)
    df["gluc_label"]         = df["gluc"].map(gluc_labels)

combined_clean.head()


,age,gender,height,weight,bmi,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,source,cholesterol_mgdl,gluc_mgdl,cholesterol_label,gluc_label
0,50,2,168,62.0,21.967120,110,80,1,1,0,0,1,0,russia,180,90,Normal (<200),Normal (<100)
1,55,1,156,85.0,34.927679,140,90,3,1,0,0,1,1,russia,270,90,Well Above Normal (>=240),Normal (<100)
2,52,1,165,64.0,23.507805,130,70,3,1,0,0,0,1,russia,270,90,Well Above Normal (>=240),Normal (<100)
3,48,2,169,82.0,28.710479,150,100,1,1,0,0,1,1,russia,180,90,Normal (<200),Normal (<100)
4,48,1,156,56.0,23.011177,100,60,1,1,0,0,0,0,russia,180,90,Normal (<200),Normal (<100)


## 6. Before / after summary

In [9]:
summary = pd.DataFrame({
    "raw_rows":  [len(russia), len(china), len(combined_raw)],
    "clean_rows":[len(russia_clean), len(china_clean), len(combined_clean)],
}, index=["russia", "china", "combined"])
summary["pct_dropped"] = ((summary["raw_rows"] - summary["clean_rows"]) / summary["raw_rows"] * 100).round(2)
summary["class_balance_cardio"] = [
    russia_clean["cardio"].mean().round(4),
    china_clean["cardio"].mean().round(4),
    combined_clean["cardio"].mean().round(4),
]
summary


,raw_rows,clean_rows,pct_dropped,class_balance_cardio
russia,70000,68598,2.00,0.4947
china,19999,19604,1.98,0.4961
combined,89999,88202,2.00,0.4950


**Result:** the cleaning pipeline drops a small, well-justified fraction of rows for physiologically-impossible
blood pressure / anthropometric values, and preserves a near-perfectly balanced target class (~50/50 `cardio`), which
means accuracy is a meaningful metric here and doesn't need class-imbalance correction (e.g. SMOTE, class weighting).

The cleaned frames produced here match `cardio_clean.csv`, `shanxi_clean.csv`, and `combined_clean.csv` in the repo
root, which are generated by the production script [`dataset.py`](../dataset.py) using identical logic. See
[`02_eda.ipynb`](02_eda.ipynb) for exploratory analysis on the cleaned data, and
[`03_model_comparison.ipynb`](03_model_comparison.ipynb) for model training and evaluation.